# EP240806a — HostPipeline quick-mode Aladin view

Run the cell below to recreate the interactive widget.

In [ ]:
import json
import astropy.units as u
from astropy.coordinates import Angle, SkyCoord
from astropy.table import Table
from ipyaladin import Aladin, EllipseError
from regions import CircleSkyRegion

config = json.loads('{"name": "EP240806a", "ra": 11.4863, "dec": 5.0941, "radius_arcsec": 20.0, "fov_deg": 0.0002777777777777778, "survey": "CDS/P/PanSTARRS/DR1/color-z-zg-g", "candidates": []}')
target = SkyCoord(config["ra"], config["dec"], unit="deg", frame="icrs")
aladin = Aladin(
    fov=config["fov_deg"],
    target=target,
    survey=config["survey"],
)

rows = config["candidates"]
if rows:
    cat_table = Table(
        rows=[
            (
                row["name"],
                row["ra"],
                row["dec"],
                row["z"],
                row["r1"],
                row["r2"],
                row["pa"],
                row["sep"],
            )
            for row in rows
        ],
        names=("Name", "RAJ2000", "DEJ2000", "z", "R1", "R2", "PA", "sep"),
    )
    cat_table["R1"].unit = u.arcsec
    cat_table["R2"].unit = u.arcsec
    cat_table["PA"].unit = u.deg
    cat_table["sep"].unit = u.arcsec
    aladin.add_table(
        cat_table,
        shape=EllipseError(
            maj_axis="R1",
            min_axis="R2",
            angle="PA",
            default_shape="cross",
        ),
        color="cyan",
    )
else:
    cat_table = Table(
        names=("Name", "RAJ2000", "DEJ2000", "z", "R1", "R2", "PA", "sep")
    )

search_circle = CircleSkyRegion(
    center=target,
    radius=Angle(config["radius_arcsec"], "arcsec"),
    visual={"edgecolor": "yellow", "linestyle": "dashed"},
)
aladin.add_graphic_overlay_from_region([search_circle])
aladin
